# Model Training

### Experimental design
- Target: `PerformanceRating`
- Predictors: all columns except `EmpNumber` and the target
- Train/test: 80/20 stratified split, random state 42
- Categorical variables: one-hot encoding
- Numerical variables: standardization in the Logistic Regression pipeline
- Class imbalance: `class_weight="balanced"`
- Models: Logistic Regression baseline and Random Forest main model
- Metrics: accuracy, balanced accuracy, macro F1, weighted F1
- Feature selection evidence: permutation importance on the held-out test set using balanced accuracy


In [29]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, balanced_accuracy_score, f1_score
from sklearn.inspection import permutation_importance


In [30]:
DATASET= r"C:\Users\karth\Desktop\cds_iabac\data\raw\INX_Future_Inc_Employee_Performance_CDS_Project2_Data_V1.8.xls"
data = pd.read_excel(DATASET)
print("Dataset loaded successfully")
data.head()

Dataset loaded successfully


,EmpNumber,Age,Gender,EducationBackground,MaritalStatus,EmpDepartment,EmpJobRole,BusinessTravelFrequency,DistanceFromHome,EmpEducationLevel,EmpEnvironmentSatisfaction,EmpHourlyRate,EmpJobInvolvement,EmpJobLevel,EmpJobSatisfaction,NumCompaniesWorked,OverTime,EmpLastSalaryHikePercent,EmpRelationshipSatisfaction,TotalWorkExperienceInYears,TrainingTimesLastYear,EmpWorkLifeBalance,ExperienceYearsAtThisCompany,ExperienceYearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager,Attrition,PerformanceRating
0,E1001000,32,Male,Marketing,Single,Sales,Sales Executive,Travel_Rarely,10,3,4,55,3,2,4,1,No,12,4,10,2,2,10,7,0,8,No,3
1,E1001006,47,Male,Marketing,Single,Sales,Sales Executive,Travel_Rarely,14,4,4,42,3,2,1,2,No,12,4,20,2,3,7,7,1,7,No,3
2,E1001007,40,Male,Life Sciences,Married,Sales,Sales Executive,Travel_Frequently,5,4,4,48,2,3,1,5,Yes,21,3,20,2,3,18,13,1,12,No,4
3,E1001009,41,Male,Human Resources,Divorced,Human Resources,Manager,Travel_Rarely,10,4,2,73,2,5,4,3,No,15,2,23,2,2,21,6,12,6,No,3
4,E1001010,60,Male,Marketing,Single,Sales,Sales Executive,Travel_Rarely,16,4,1,84,3,2,1,8,No,14,4,10,1,3,2,2,2,2,No,3


In [31]:
X = data.drop(columns =["EmpNumber","PerformanceRating"])
Y = data["PerformanceRating"]

In [38]:
#Defining columner Trasformer
cat = X.select_dtypes(include="object").columns.tolist()
num = X.select_dtypes(exclude="object").columns.tolist()
pre = ColumnTransformer([("cat",OneHotEncoder(handle_unknown="ignore"),cat),
                            ("num",StandardScaler(), num)])

In [39]:
x_train, x_test, y_train, y_test = train_test_split(X,Y, test_size=0.2,stratify=Y, random_state=42)

In [41]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1500,class_weight="balanced"),
    "Random Forest": RandomForestClassifier(n_estimators=500,class_weight="balanced",random_state=42, min_samples_leaf=2,n_jobs=-1)
}
fitted={}
rows=[]
for name, model in models.items():
    pipe = Pipeline([("preprocess",pre),("model",model)])
    pipe.fit(x_train, y_train)
    pred=pipe.predict(x_test)
    fitted[name]=pipe
    rows.append([name,accuracy_score(y_test,pred), balanced_accuracy_score(y_test,pred),
                f1_score(y_test,pred,average="macro"),f1_score(y_test,pred,average="weighted")])

In [44]:
pd.DataFrame(rows,columns=["Model","Accuracy","Balanced Accuracy","Macro F1","Weighted F1"]).round(4).sort_values("Accuracy",ascending=False)

,Model,Accuracy,Balanced Accuracy,Macro F1,Weighted F1
1,Random Forest,0.9208,0.8542,0.8709,0.9195
0,Logistic Regression,0.7583,0.7576,0.6869,0.7730


In [48]:
rf = fitted["Random Forest"]
pred = rf.predict(x_test)
print(classification_report(y_test,pred,digits=4))
print("Confusion matrix:")
print(confusion_matrix(y_test,pred))

              precision    recall  f1-score   support

           2     0.8718    0.8718    0.8718        39
           3     0.9385    0.9600    0.9492       175
           4     0.8636    0.7308    0.7917        26

    accuracy                         0.9208       240
   macro avg     0.8913    0.8542    0.8709       240
weighted avg     0.9196    0.9208    0.9195       240

Confusion matrix:
[[ 34   5   0]
 [  4 168   3]
 [  1   6  19]]


In [80]:
perm = permutation_importance(rf,x_test,y_test,scoring="balanced_accuracy",n_repeats=20,random_state=42,n_jobs=-1)
importance = pd.DataFrame(perm.importances_mean,index=x_test.columns,columns=["Mean permutation importance"])

In [81]:
display(importance.sort_values(by="Mean permutation importance", ascending=False).head(10))

,Mean permutation importance
EmpEnvironmentSatisfaction,0.242476
EmpLastSalaryHikePercent,0.232864
YearsSinceLastPromotion,0.085028
ExperienceYearsInCurrentRole,0.033719
EmpDepartment,0.022521
EmpJobRole,0.008339
YearsWithCurrManager,0.001164
EducationBackground,0.000571
TrainingTimesLastYear,0.000286
Attrition,0.000190


In [82]:
importance_df = importance.reset_index()
importance_df.columns = ["Feature", "Mean permutation importance"]
importance_df = importance_df.sort_values(by="Mean permutation importance",ascending=False)

In [77]:
importance_df.to_csv(r"C:\Users\karth\Desktop\cds_iabac\project summary\Analysis\feature_importance.csv", index=False)
print("feature importance dataset saved.")


feature importance dataset saved.
